<div align="center">

<h1>Buzz in Social Media</h1>

<h3>Analyse du dataset UCI <em>Buzz in Social Media</em></h3>

<p><b>Côme Rossary</b> &nbsp;&middot;&nbsp; <b>Batiste Fancellu</b></p>

<p>4 novembre 2026</p>

</div>

---

## Objectif

L'objectif de ce devoir et de ce notebook est d'extraire le maximum d'information pertinente du dataset « Buzz in Social Media », créé par François Kawala, Ahlame Douzal, Eric Gaussier et Eustache Diemert (Université Joseph Fourier / BestofMedia Group) et mis en ligne sur le dépôt UCI en mai 2013.

Ce dataset suit l'activité autour de sujets sur deux réseaux sociaux différents :
- Twitter ;
- Tom's Hardware, un forum consacré aux nouvelles technologies et ayant des dynamiques plus conservatrices.

Avant d'aller plus loin, importons ce dataset.

In [1]:
import os

# Sur Google Colab : téléchargement du dossier du projet depuis Google Drive.
# En local : on suppose être déjà dans le dossier du projet.
if os.path.exists("/content"):
    if not os.path.exists("/content/projet"):
        !pip install -q gdown
        !gdown -q --folder 1oaoxufMxfSdoyX4hrLdCGesuG3NUDZ57 -O /content/projet
    %cd /content/projet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

## Les fichiers du dataset

Le dataset contient un dossier par réseau social, `Twitter` et `TomsHardware`, avec deux fichiers chacun :

- `*.data` : les données, au format CSV sans en-tête, une ligne par sujet ;
- `*.names` : la documentation du fichier `.data`, au format texte habituel des datasets UCI.

### Ce que nous apprend le fichier `.names`

Pour les deux datasets, chaque ligne correspond à un sujet (un exemple). Les attributs sont des indicateurs d'activité mesurés à plusieurs pas de temps consécutifs. La dernière colonne est la valeur à prédire.

**Dossier `Twitter`** : chaque ligne est composée de 77 attributs, soit 11 indicateurs × 7 pas de temps (t = 0 à 6), dans cet ordre :

| Colonnes | Indicateur | Signification |
|---|---|---|
| 0 à 6 | NCD, *Number of Created Discussions* | nombre de discussions créées sur le sujet |
| 7 à 13 | AI, *Author Increase* | nombre de nouveaux auteurs sur le sujet |
| 14 à 20 | AS(NA), *Attention Level (authors)* | part des auteurs du réseau qui écrivent sur le sujet |
| 21 à 27 | BL, *Burstiness Level* | NCD / NAD : part des discussions actives qui viennent d'être créées |
| 28 à 34 | NAC, *Number of Atomic Containers* | nombre de contributions (tweets) sur le sujet |
| 35 à 41 | AS(NAC), *Attention Level (contributions)* | part des contributions du réseau qui concernent le sujet |
| 42 à 48 | CS, *Contribution Sparseness* | dispersion des contributions entre les discussions du sujet |
| 49 à 55 | AT, *Author Interaction* | nombre moyen d'auteurs par discussion |
| 56 à 62 | NA, *Number of Authors* | nombre d'auteurs actifs sur le sujet |
| 63 à 69 | ADL, *Average Discussion Length* | longueur moyenne d'une discussion |
| 70 à 76 | NAD, *Number of Active Discussions* | nombre de discussions actives sur le sujet |
| 77 | cible | nombre moyen de discussions actives (NAD) dans les semaines suivantes |

**Dossier `TomsHardware`** : chaque ligne est composée de 96 attributs, soit 12 indicateurs × 8 pas de temps (t = 0 à 7). Ce sont les mêmes indicateurs dans un ordre différent, plus un indicateur propre au forum :

| Colonnes | Indicateur |
|---|---|
| 0 à 7 | NCD |
| 8 à 15 | BL |
| 16 à 23 | NAD |
| 24 à 31 | AI |
| 32 à 39 | NAC, ici le nombre de messages postés |
| 40 à 47 | ND, *Number of Displays* : nombre d'affichages des discussions du sujet |
| 48 à 55 | CS |
| 56 à 63 | AT |
| 64 à 71 | NA |
| 72 à 79 | ADL |
| 80 à 87 | AS(NA) |
| 88 à 95 | AS(NAC) |
| 96 | cible : nombre moyen d'affichages (ND) dans les semaines suivantes |

Commençons par nous assurer que le dataset est complet : nombre de colonnes annoncé par le `.names`, pas de valeurs manquantes, colonnes bien numériques, etc.

In [2]:
from pathlib import Path

# Indicateurs dans l'ordre des colonnes de chaque fichier .data, et nombre de pas de temps (cf. .names).
DATASETS = {
    "Twitter": {
        "n_steps": 7,
        "indicateurs": ["NCD", "AI", "AS(NA)", "BL", "NAC", "AS(NAC)", "CS", "AT", "NA", "ADL", "NAD"],
    },
    "TomsHardware": {
        "n_steps": 8,
        "indicateurs": ["NCD", "BL", "NAD", "AI", "NAC", "ND", "CS", "AT", "NA", "ADL", "AS(NA)", "AS(NAC)"],
    },
}


def charger(nom):
    """Charge <nom>.data (cherché depuis le dossier courant) et nomme les colonnes X_t ; la dernière est 'cible'."""
    fichiers = list(Path(".").rglob(f"{nom}.data"))
    assert fichiers, f"{nom}.data introuvable depuis {Path.cwd()}"
    spec = DATASETS[nom]
    colonnes = [f"{ind}_{t}" for ind in spec["indicateurs"] for t in range(spec["n_steps"])] + ["cible"]
    df = pd.read_csv(fichiers[0], header=None)
    assert df.shape[1] == len(colonnes), f"{nom} : {df.shape[1]} colonnes au lieu de {len(colonnes)}"
    df.columns = colonnes
    return df


data = {nom: charger(nom) for nom in DATASETS}

for nom, df in data.items():
    non_num = (~df.dtypes.apply(pd.api.types.is_numeric_dtype)).sum()
    print(f"{nom:13s} {df.shape[0]:>8,} lignes x {df.shape[1]} colonnes | "
          f"valeurs manquantes : {df.isna().sum().sum()} | "
          f"colonnes non numériques : {non_num} | "
          f"lignes dupliquées : {df.duplicated().sum():,}")

Twitter        583,250 lignes x 78 colonnes | valeurs manquantes : 0 | colonnes non numériques : 0 | lignes dupliquées : 6,016
TomsHardware    28,179 lignes x 97 colonnes | valeurs manquantes : 0 | colonnes non numériques : 0 | lignes dupliquées : 25
